In [0]:
# =============================================================================
# NOTEBOOK  : gold_dim_store
# PURPOSE   : silver.store_master → gold.dim_store
# LAYER     : Gold (dimension table)
# FREQUENCY : Weekly (after silver_store_master completes)
# WRITE     : Full overwrite — dim table, small (500 rows), always current state
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import GOLD_DIM_STORE_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col, round, when, datediff, current_date
)
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["silver_store_master"]
TARGET_TABLE = cfg["tables"]["gold_dim_store"]
PIPELINE     = "gold_dim_store"

In [0]:
# ── Read + Transform + Write ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    df = (
        spark.read.table(SOURCE_TABLE)
        .filter(col("is_current") == True)
    )
 
    rows_read = df.count()
    print(f"[READ] {rows_read} current store records")
 
    df = (
        df
        .withColumn("years_in_operation",
            round(datediff(current_date(), col("opening_date")) / 365.25, 1))
        .withColumn("store_size_tier",
            when(col("store_type") == "Warehouse",  "Large")
            .when(col("store_type") == "Superstore", "Medium")
            .otherwise("Small"))
        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
        .select([f.name for f in GOLD_DIM_STORE_SCHEMA.fields])
    )
 
    rows_written = df.count()
 
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )
 
    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.read.table(TARGET_TABLE).groupBy("store_size_tier", "region").count().show()